### Import dependencies

In [39]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

### create a fully connected neural

In [40]:
class NN(nn.Module):
    def __init__ (self, input_size, num_class):
        super().__init__()
        self.fc1 = nn.Linear(input_size,50)
        self.fc2 = nn.Linear(50,num_class)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### SET A DEVICE

In [41]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Hyperparameter

In [42]:
input_size = 784
num_classes = 10
learning_rate = 0.001
batch_size = 64
num_epochs = 10

### Load data

In [43]:
train_dataset = datasets.MNIST(root='datasets/', train = True, transform = transforms.ToTensor(), download = True)
train_loader = DataLoader(dataset=train_dataset, batch_size = batch_size, shuffle = True)

test_dataset = datasets.MNIST(root='datasets/', train = False, transform = transforms.ToTensor(), download = True)
test_loader = DataLoader(dataset=test_dataset, batch_size = batch_size, shuffle = True)

### Initialize network

In [ ]:
model = NN(input_size = input_size, num_class= num_classes).to(device)

### Loss and Optimizer

In [45]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)


### Train network

In [ ]:
for epoch in range(num_epochs):
    losses = []  # collect losses for this epoch
    
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device=device)
        targets = targets.to(device=device)
        data = data.reshape(data.shape[0], -1)

        # forward
        scores = model(data)
        loss = criterion(scores, targets)

        # backward
        optimizer.zero_grad()   # 1. clear old gradients
        loss.backward()         # 2. calculate new gradients
        optimizer.step()        # 3. update the weights
        
        losses.append(loss.item())  # save each batch loss
    
    # print after each epoch
    mean_loss = sum(losses) / len(losses)
    print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {mean_loss:.4f}')

Epoch [1/10] Loss: 0.4233
Epoch [2/10] Loss: 0.2211
Epoch [3/10] Loss: 0.1716
Epoch [4/10] Loss: 0.1414
Epoch [5/10] Loss: 0.1197
Epoch [6/10] Loss: 0.1041
Epoch [7/10] Loss: 0.0923
Epoch [8/10] Loss: 0.0819
Epoch [9/10] Loss: 0.0743
Epoch [10/10] Loss: 0.0678


### Check accuracy on training and test

In [48]:
def check_acc(loader, model):
    if loader.dataset.train:
        print("Checking accuracy on training data")
    else:
        print("Checking accuracy on test data")
    
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)
            x = x.reshape(x.shape[0], -1)

            scores = model(x)
            _, prediction = scores.max(1)
            num_correct += (prediction == y).sum()
            num_samples += prediction.size(0)

    acc = float(num_correct) / float(num_samples) * 100
    print(f"Got {num_correct} / {num_samples} with accuracy {acc:.2f}%")

    model.train()
    return acc

check_acc(train_loader, model)
check_acc(test_loader, model)

Checking accuracy on training data
Got 58990 / 60000 with accuracy 98.32%
Checking accuracy on test data
Got 9715 / 10000 with accuracy 97.15%


97.15